In [1]:
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix # Added import

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the dataset
dataset = load_dataset("ailsntua/QEvasion")

# Prepare labels
labels = dataset['train'].unique('clarity_label')
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

def add_labels(example):
    example['labels'] = label2id[example['clarity_label']]
    return example

dataset = dataset.map(add_labels)
dataset = dataset.remove_columns([
    col for col in dataset['train'].column_names if col not in ['question', 'interview_answer', 'labels']
])

print("Dataset ready:")
print(dataset)
print(f"Labels mapped: {label2id}")


# Calculate class weights for imbalanced data
def get_class_weights(dataset, num_labels):
    label_counts = Counter(dataset["train"]["labels"])
    total_samples = len(dataset["train"])

    # Calculate class weights (inverse frequency)
    class_weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)  # avoid division by zero
        weight = total_samples / (num_labels * count)
        class_weights.append(weight)

    # Move the tensor to the active device (GPU)
    return torch.tensor(
        class_weights, dtype=torch.float32, device=device
    )


# Get class weights
class_weights = get_class_weights(dataset, num_labels)
print(f"Class weights: {class_weights}")
print(f"Class weights device: {class_weights.device}")  # Verify it's on CUDA

# Focal Loss implementation :cite[1]:cite[8]
class FocalLoss(nn.Module):
    """
    Multi-class Focal loss implementation
    Focal loss helps address class imbalance by focusing on hard examples

    Args:
        gamma (float): Focusing parameter, higher values put more focus on hard examples
        weight (Tensor): Class weights tensor for handling imbalanced data
        ignore_index (int): Index to ignore in loss calculation
    """
    def __init__(self, gamma=1.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, input, target):
        # Calculate cross entropy loss
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight, ignore_index=self.ignore_index)

        # Get probabilities
        pt = torch.exp(-ce_loss)

        # Compute focal loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()


# Updated Custom Trainer with corrected compute_loss signature
class CustomTrainer(Trainer):
    """
    Custom trainer that uses Focal Loss with class weights
    This subclass overrides the compute_loss method to use our custom loss function
    """

    def __init__(self, *args, class_weights=None, focal_gamma=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=focal_gamma, weight=class_weights)

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        # Extract labels and run model forward pass
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Compute focal loss with class weights
        loss = self.focal_loss(logits, labels)

        # Handle return_outputs as required by the Trainer
        return (loss, outputs) if return_outputs else loss

# Tokenization and model setup
model_checkpoint = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples['question'],
        examples['interview_answer'],
        truncation=True,
        padding="max_length",
        max_length=1680
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    # Use macro averaging for balanced metrics across classes
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    # Keep weighted for comparison
    precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted
    }

# --- MODIFICATION 1: Updated TrainingArguments ---
# We now evaluate, save, and load the best model based on 'f1_macro'
training_args = TrainingArguments(
    output_dir="ModernBERT_QEvasion_model",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",          # <--- MODIFIED (was "no")
    save_strategy="epoch",
    load_best_model_at_end=True,    # <--- MODIFIED (was False)
    metric_for_best_model="f1_macro", # <--- NEW
    greater_is_better=True,         # <--- NEW
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
    fp16=True,  # Enable mixed precision (reduces memory usage)
    gradient_checkpointing=True,
)

# --- MODIFICATION 2: Updated CustomTrainer instantiation ---
# We pass the test set to eval_dataset
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],  # <--- NEW
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,  # Pass the calculated class weights
    focal_gamma=1.0,  # You can adjust this parameter
)

print(f"Using class weights: {class_weights}")
print("Starting training with Focal Loss (evaluating on test set after each epoch)...")
trainer.train()

print("Training completed!")

# --- MODIFICATION 3: Updated Final Evaluation ---
# This will now evaluate the *best* model saved during training
# (because of load_best_model_at_end=True)
test_results = trainer.evaluate() # <--- MODIFIED (no arg needed)
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (from best epoch: {trainer.state.best_model_checkpoint})") # <--- MODIFIED
print("="*60)
for key, value in test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# Optional: Get detailed predictions
# This will also use the best model
print("\nDetailed predictions analysis (from best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                          target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3597412039.py:98: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Using class weights: tensor([1.0925, 0.5634, 3.2285], device='cuda:0')
Starting training with Focal Loss (evaluating on test set after each epoch)...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
1,0.665200,0.603229,0.269481,0.255066,0.132097,0.195238,0.091790,0.442671,0.269481
2,0.571000,0.559935,0.529221,0.502083,0.545648,0.487552,0.625720,0.602282,0.529221
3,0.374500,0.480238,0.519481,0.518482,0.529640,0.508005,0.628372,0.625910,0.519481
4,0.229900,0.585625,0.633117,0.562147,0.638601,0.537063,0.651176,0.612573,0.633117
5,0.144300,0.950764,0.646104,0.580103,0.652666,0.586879,0.665232,0.580685,0.646104


Training completed!



FINAL TEST RESULTS (from best epoch: ModernBERT_QEvasion_model/checkpoint-2155)
eval_loss: 0.9508
eval_accuracy: 0.6461
eval_f1_macro: 0.5801
eval_f1_weighted: 0.6527
eval_precision_macro: 0.5869
eval_precision_weighted: 0.6652
eval_recall_macro: 0.5807
eval_recall_weighted: 0.6461

Detailed predictions analysis (from best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.45      0.57      0.50        79
     Ambivalent       0.76      0.69      0.73       206
Clear Non-Reply       0.55      0.48      0.51        23

       accuracy                           0.65       308
      macro avg       0.59      0.58      0.58       308
   weighted avg       0.67      0.65      0.65       308


Confusion Matrix:
[[ 45  34   0]
 [ 54 143   9]
 [  1  11  11]]


In [2]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 10
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))



STARTING PART 2: Resuming training for 5 additional epochs...

Updated total number of epochs to: 10


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
6,0.178900,0.814740,0.662338,0.566222,0.657731,0.556907,0.659196,0.590982,0.662338
7,0.124200,1.159084,0.652597,0.573794,0.658614,0.569947,0.668455,0.581320,0.652597
8,0.037800,1.511711,0.642857,0.568899,0.648947,0.584483,0.661247,0.563591,0.642857
9,0.014000,1.938907,0.662338,0.556395,0.655164,0.575483,0.650759,0.542085,0.662338
10,0.017400,1.912630,0.659091,0.572611,0.662266,0.583494,0.667753,0.566479,0.659091


Additional 5 epochs of training completed!

FINAL TEST RESULTS (after 10 total epochs)
(Best model from all runs: ModernBERT_QEvasion_model/checkpoint-2155)


eval_loss: 0.9508
eval_accuracy: 0.6461
eval_f1_macro: 0.5801
eval_f1_weighted: 0.6527
eval_precision_macro: 0.5869
eval_precision_weighted: 0.6652
eval_recall_macro: 0.5807
eval_recall_weighted: 0.6461

Detailed predictions analysis (from new best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.45      0.57      0.50        79
     Ambivalent       0.76      0.69      0.73       206
Clear Non-Reply       0.55      0.48      0.51        23

       accuracy                           0.65       308
      macro avg       0.59      0.58      0.58       308
   weighted avg       0.67      0.65      0.65       308


Confusion Matrix:
[[ 45  34   0]
 [ 54 143   9]
 [  1  11  11]]
